<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.lab.1. Wisconsin Cancer Classification - Neural Networks
  </strong>
</div>

In [ ]:
import os
import time

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rich.console import Console
from rich.table import Table
from tqdm import tqdm
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

console = Console()

- **Explanation:** Here, all the libraries and methods necessary for the project are being imported, both for data manipulation, visualization, image handling, and for the training flow with neural networks.
- **Improvement:** What I did was remove the dependencies that I saw were not being used in the project, and add rich.table and rich.console to improve the visualization when printing the results.

Checking if GPU is available

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
console.print(f"Using device: {device}")

- **Explanation:** This validates whether a GPU is available in the execution environment. If so, cuda:0 is assigned; otherwise, cpu is assigned.
- **Improvement:** Change print to console.print to print messages in a more professional manner.

In [ ]:
DATA_DIR = "./data/Pneumonia/chest_xray"

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

CLASSES = ["NORMAL", "PNEUMONIA"]


def count_images(directory, CLASSES):
    counts = {}
    for cls in CLASSES:
        class_path = os.path.join(directory, cls)
        counts[cls] = len(os.listdir(class_path)) if os.path.exists(class_path) else 0
    return counts


def show_dataset_counts(train_counts, val_counts, test_counts):

    table = Table(title="Dataset Structure")

    table.add_column("Set", justify="center", style="cyan")
    table.add_column("Class", justify="center", style="green")
    table.add_column("Image Count", justify="center", style="green")

    for cls in CLASSES:
        table.add_row("Training", cls, str(train_counts.get(cls)))
        table.add_row("Test", cls, str(test_counts.get(cls)))
        table.add_row("Validation", cls, str(val_counts.get(cls)))

    console.print(table)

In [ ]:
train_counts = count_images(TRAIN_DIR, CLASSES)
val_counts = count_images(VAL_DIR, CLASSES)
test_counts = count_images(TEST_DIR, CLASSES)

show_dataset_counts(train_counts, val_counts, test_counts)

- **Explanation:**
Here, the main path of the dataset is defined and the training, validation, and test directories are organized. Then, using the count_images function, each class is scanned to count how many images there are, and the image counts for each dataset set are displayed.

- **Improvement:**
What I did was capitalize the variables that function as constants, and I optimized the validation within the count_images for loop to make it clearer and more readable. Finally, I encapsulated the visualization in a method that displays the data in table format instead of simple print statements.

Calculating class distribution: 

In [ ]:
def plot_class_distribution(class_counts, title="Class Distribution"):
    total = sum(class_counts.values())

    distribution = {cls: (count / total * 100) for cls, count in class_counts.items()}

    counts = list(class_counts.values())
    percentages = [distribution[cls] for cls in CLASSES]

    plt.figure(figsize=(10, 6))
    sns.barplot(x=CLASSES, y=counts)
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Number of Images")

    for i, (count, pct) in enumerate(zip(counts, percentages)):
        plt.text(i, count, f"{count} ({pct:.1f}%)", ha="center", va="bottom")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_class_distribution(train_counts, title="Class Distribution in Training Set")

- **Explanation:**
Here, we calculate how many images there are in total in the training set and then use a bar chart to visually display that distribution.

- **Improvement:**
What I did was move everything to a method so that the code would be more organized. I also used zip to combine the counts and percentages in the same cycle, to make it clearer and easier to see what value is being added to each bar in the chart.

In [ ]:
def show_samples(directory, n_samples=3):
    num_classes = len(CLASSES)
    num_cols = n_samples
    num_rows = num_classes

    fig, axs = plt.subplots(num_rows, num_cols, figsize=(4 * num_cols, 4 * num_rows))

    for row_idx, cls in enumerate(CLASSES):
        class_path = os.path.join(directory, cls)
        if not os.path.exists(class_path):
            continue

        image_files = os.listdir(class_path)[:n_samples]
        for col_idx, img_file in enumerate(image_files):
            img_path = os.path.join(class_path, img_file)
            img = Image.open(img_path).convert("L")
            axs[row_idx, col_idx].imshow(img, cmap="gray")
            axs[row_idx, col_idx].set_title(cls)
            axs[row_idx, col_idx].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_samples(TRAIN_DIR)

- **Explanation:**
Here, sample images are shown for each class in the dataset, in this case three per class (n_samples=3). Each image is loaded and drawn on a grid, and the axes are hidden so that only the images are displayed.

- **Improvement:**
What I did was separate the number of rows, columns, and classes into variables, so it would be clearer where those values come from in the subplot. 

In [ ]:
class ChestXRayDataset(Dataset):
    def __init__(self, root_dir, transform=None, grayscale=True):
        self.root_dir = root_dir
        self.transform = transform
        self.grayscale = grayscale

        self.image_paths = []
        self.labels = []

        valid_extensions = (".png", ".jpg", ".jpeg")

        for i, cls_name in enumerate(CLASSES):
            class_dir = os.path.join(root_dir, cls_name)
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(valid_extensions):
                    img_path = os.path.join(class_dir, img_name)
                    self.image_paths.append(img_path)
                    self.labels.append(i)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        image = Image.open(img_path)

        image = image.convert("L" if self.grayscale else "RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return image, label

- **Explanation:** Here, a class is defined that represents the complete dataset of a specific set, whether train, test, or val. Within it, the paths of all images are loaded along with their labels (0 for NORMAL and 1 for PNEUMONIA). The __len__ method returns the total number of images in the set, while __getitem__ allows access to a specific image.

- **Improvement:**
What I did was remove the repeated code when loading the images for each class, making it cleaner and easier to read. I also improved the if statement in the use of convert to make it simpler and cleaner.

In [ ]:
def create_dataset(directory, transform, grayscale=True):
    return ChestXRayDataset(directory, transform, grayscale)


def create_loader(dataset, batch_size=32, shuffle=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [ ]:
transform_direct = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)

train_dataset_direct = create_dataset(TRAIN_DIR, transform_direct)
val_dataset_direct = create_dataset(VAL_DIR, transform_direct)
test_dataset_direct = create_dataset(TEST_DIR, transform_direct)

batch_size = 32

train_loader_direct = create_loader(train_dataset_direct, batch_size, shuffle=True)
val_loader_direct = create_loader(val_dataset_direct, batch_size)
test_loader_direct = create_loader(test_dataset_direct, batch_size)

- **Explanation:** Here, the transformations to be applied to the images are defined (resize, convert to tensor, and normalize). Then, using the ChestXRayDataset class, the datasets are prepared with the transformations and then the dataloaders are created, which allow the data to be loaded in batches of a defined size (batch_size) and, in the case of training (shuffle=True).

- **Improvement:** What I did was create two separate methods, one to build the datasets and another to generate the dataloaders to eliminate the need for comments, and I also reduced the duplicate code, making it clearer and easier to read.

In [ ]:
def extract_direct_features(dataloader):
    with torch.no_grad():
        all_features, all_labels = process_dataloader(dataloader)

    return np.vstack(all_features), np.array(all_labels)


def process_dataloader(dataloader):
    all_features = []
    all_labels = []

    for images, labels in tqdm(dataloader, desc="Extracting direct features"):

        batch_size = images.size(0)
        features = images.view(batch_size, -1).numpy()

        labels_numpy = labels.numpy()

        all_features.append(features)
        all_labels.extend(labels_numpy)

    return all_features, all_labels


X_train_direct, y_train_direct = extract_direct_features(train_loader_direct)
X_val_direct, y_val_direct = extract_direct_features(val_loader_direct)
X_test_direct, y_test_direct = extract_direct_features(test_loader_direct)

console.print(f"Direct features shape (training): {X_train_direct.shape}")

- **Explanation:** Here, the characteristics of the images are extracted by going through each batch of the dataloader. The images are transformed into vectors and saved together with their labels for later use in training.

- **Improvement:** I separated the logic into a new method (process_dataloader) to make the code cleaner and easier to understand. I also added console.print instead of print to display the data format more clearly.

In [ ]:
def load_alexnet_model(device):
    alexnet = models.alexnet(pretrained=True)
    alexnet = alexnet.to(device)
    alexnet.eval()
    return alexnet


transform_alexnet = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.size(0) == 1 else x),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_dataset_alexnet = create_dataset(TRAIN_DIR, transform_direct, grayscale=False)
test_dataset_alexnet = create_dataset(TEST_DIR, transform_direct, grayscale=False)
val_dataset_alexnet = create_dataset(VAL_DIR, transform_direct, grayscale=False)

batch_size = 32
train_loader_alexnet = create_loader(
    train_dataset_alexnet, batch_size=batch_size, shuffle=True
)
val_loader_alexnet = create_loader(val_dataset_alexnet, batch_size=batch_size)
test_loader_alexnet = create_loader(test_dataset_alexnet, batch_size=batch_size)

alexnet = load_alexnet_model(device)
console.print(alexnet)

- **Explanation:** Here we define the transformations that AlexNet needs, such as resizing, normalizing, and repeating when the image is grayscale. Then we create the datasets and dataloaders for training, validation, and testing, and load the pre-trained AlexNet model.

- **Improvement:** I organized the code with the functions I had already created for the direct model (create_dataset, create_loader) and created a load_alexnet_model function to make it cleaner and easier to read. 

In [ ]:
def extract_alexnet_features(dataloader):
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Extracting AlexNet features"):
            images = images.to(device)

            # Forward pass through AlexNet until the last fully connected layer
            # This gives us a 4096-dimensional feature vector
            features = alexnet.features(images)
            features = alexnet.avgpool(features)
            features = torch.flatten(features, 1)
            features = alexnet.classifier[0](features)  # ReLU
            features = alexnet.classifier[1](features)  # Dropout
            features = alexnet.classifier[2](features)  # Linear
            features = alexnet.classifier[3](features)  # ReLU
            features = alexnet.classifier[4](features)  # Dropout
            features = alexnet.classifier[5](features)  # Final 4096-dim features

            features = features.cpu().numpy()
            all_features.append(features)
            all_labels.extend(labels.numpy())

    return np.vstack(all_features), np.array(all_labels)


X_train_alexnet, y_train_alexnet = extract_alexnet_features(train_loader_alexnet)
X_val_alexnet, y_val_alexnet = extract_alexnet_features(val_loader_alexnet)
X_test_alexnet, y_test_alexnet = extract_alexnet_features(test_loader_alexnet)

console.print(f"AlexNet features shape (training): {X_train_alexnet.shape}")

- **Explanation:** Here, the characteristics of the images are extracted using AlexNet. Basically, a forward pass is made through the layers of the network until reaching the last vector of 4096 dimensions.

- **Improvement:** I didn't make any changes because the code is already clear and well documented. The only thing I modified was to use console.print instead of print to maintain the format in the outputs.

- **Why I consider it to be AI:** When reviewing this block, the first thing I noticed was how detailed it was, both in the code and in the comments. I tried passing the following prompt to the AI we were told about, taking one of those comments into account, and the response was very similar code that, as I asked for more detail, got even closer to what was already done:

**Prompt:** `"Extrae los features de las imágenes proporcionadas usando la arquitectura AlexNet, realiza un forward pass hasta la última capa completamente conectada."`

This gave me a hint that there is a high probability that it was generated with the help of AI.

In [ ]:
def scale_features(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled


X_train_direct_scaled, X_val_direct_scaled, X_test_direct_scaled = scale_features(
    X_train_direct, X_val_direct, X_test_direct
)
X_train_alexnet_scaled, X_val_alexnet_scaled, X_test_alexnet_scaled = scale_features(
    X_train_alexnet, X_val_alexnet, X_test_alexnet
)

- **Explanation:** Here, the training, validation, and test data are normalized with StandardScaler so that all features are on the same scale.

- **Improvement:** I didn't make any; the code is already clear and easy to understand.

In [ ]:
def train_model(X_train, y_train, max_iter=1000):
    model = MLPClassifier(
        hidden_layer_sizes=(100, 50),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        max_iter=max_iter,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
    )

    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    return model, training_time


def evaluate_model(model, X_val, X_test):
    val_predictions = model.predict(X_val)
    test_predictions = model.predict(X_test)

    return val_predictions, test_predictions


def calculate_metrics(
    model, y_val, val_predictions, y_test, test_predictions, training_time
):
    val_accuracy = accuracy_score(y_val, val_predictions)
    test_accuracy = accuracy_score(y_test, test_predictions)
    test_precision = precision_score(y_test, test_predictions)
    test_recall = recall_score(y_test, test_predictions)
    test_f1 = f1_score(y_test, test_predictions)

    conf_matrix = confusion_matrix(y_test, test_predictions)
    class_report = classification_report(y_test, test_predictions, target_names=CLASSES)

    return {
        "training_time": training_time,
        "val_accuracy": val_accuracy,
        "test_accuracy": test_accuracy,
        "test_precision": test_precision,
        "test_recall": test_recall,
        "test_f1": test_f1,
        "confusion_matrix": conf_matrix,
        "classification_report": class_report,
        "loss_curve": model.loss_curve_,
    }


def train_and_evaluate_model(
    X_train, y_train, X_val, y_val, X_test, y_test, max_iter=1000
):
    model, training_time = train_model(X_train, y_train, max_iter)

    val_predictions, test_predictions = evaluate_model(model, X_val, X_test)

    results = calculate_metrics(
        model, y_val, val_predictions, y_test, test_predictions, training_time
    )

    return results


def create_classification_comparison_table(title, results):
    table = Table(title=title)

    table.add_column("Model", justify="center", style="cyan")
    table.add_column("Train Time (s)", justify="right", style="green")
    table.add_column("Val Accuracy", justify="right", style="green")
    table.add_column("Test Accuracy", justify="right", style="green")
    table.add_column("Test Precision", justify="right", style="green")
    table.add_column("Test Recall", justify="right", style="green")
    table.add_column("Test F1", justify="right", style="green")

    for model_name, metrics in results.items():
        table.add_row(
            model_name,
            f'{metrics["training_time"]:.4f}',
            f'{metrics["val_accuracy"]:.4f}',
            f'{metrics["test_accuracy"]:.4f}',
            f'{metrics["test_precision"]:.4f}',
            f'{metrics["test_recall"]:.4f}',
            f'{metrics["test_f1"]:.4f}',
        )

    console.print(table)

- **Explanation:** Here, two different models are trained, one with a direct approach and the other using transfer learning with AlexNet, both with MLPClassifier. In addition, the metrics and confusion matrix are obtained, and the training time and loss curve are saved.

- **Improvement:** I separated the logic into methods to clarify responsibilities and created a method to display a comparative table of metrics by model.

In [ ]:
CLASSIFIER_A = "Classifier A (Direct Approach)"
CLASSIFIER_B = "Classifier B (Transfer Learning with AlexNet)"

results_direct = train_and_evaluate_model(
    X_train_direct_scaled,
    y_train_direct,
    X_val_direct_scaled,
    y_val_direct,
    X_test_direct_scaled,
    y_test_direct,
)

results_alexnet = train_and_evaluate_model(
    X_train_alexnet_scaled,
    y_train_alexnet,
    X_val_alexnet_scaled,
    y_val_alexnet,
    X_test_alexnet_scaled,
    y_test_alexnet,
)

all_results = {CLASSIFIER_A: results_direct, CLASSIFIER_B: results_alexnet}

create_classification_comparison_table(f"{CLASSIFIER_A} vs {CLASSIFIER_B}", all_results)

- **Explanation:** Here we had a printout of the results of the metrics obtained for each model.

- **Improvement:** What I did was use the method created previously so that instead of showing printouts, it would show a comparative table.

In [ ]:
def plot_confusion_matrices(cm1, cm2, title1, title2):

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    sns.heatmap(
        cm1,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASSES,
        yticklabels=CLASSES,
        ax=axes[0],
    )
    axes[0].set_title(title1)
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")

    sns.heatmap(
        cm2,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASSES,
        yticklabels=CLASSES,
        ax=axes[1],
    )
    axes[1].set_title(title2)
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")

    plt.tight_layout()
    plt.show()


plot_confusion_matrices(
    results_direct["confusion_matrix"],
    results_alexnet["confusion_matrix"],
    CLASSIFIER_A,
    CLASSIFIER_B,
)

- **Explanation:** The confusion matrices of the two models are plotted here, allowing them to be compared in terms of their TP, FP, TN, and FN.

- **Improvement:** None, the code block is already clear and performs its function well. 

In [ ]:
def plot_comparison_bar_chart(results_direct, results_alexnet, metrics, metric_names):
    x = np.arange(len(metrics))
    width = 0.35

    plt.figure(figsize=(12, 6))

    bars_a = plt.bar(
        x - width / 2, [results_direct[m] for m in metrics], width, label=CLASSIFIER_A
    )

    for i, bar in enumerate(bars_a):
        value = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.01,
            f"{value:.2f}" if metrics[i] == "training_time" else f"{value:.4f}",
            ha="center",
            va="bottom",
        )

    bars_b = plt.bar(
        x + width / 2, [results_alexnet[m] for m in metrics], width, label=CLASSIFIER_B
    )

    for i, bar in enumerate(bars_b):
        value = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.01,
            f"{value:.2f}" if metrics[i] == "training_time" else f"{value:.4f}",
            ha="center",
            va="bottom",
        )

    plt.xticks(x, [metric_names[m] for m in metrics], rotation=45)
    plt.ylabel("Score")
    plt.title("Comparison of Classifier Performance")
    plt.legend()
    plt.tight_layout()
    plt.show()


metric_names = {
    "training_time": "Training Time (s)",
    "val_accuracy": "Validation Accuracy",
    "test_accuracy": "Test Accuracy",
    "test_precision": "Test Precision",
    "test_recall": "Test Recall",
    "test_f1": "Test F1 Score",
}

metrics = list(metric_names.keys())

plot_comparison_bar_chart(results_direct, results_alexnet, metrics, metric_names)

- **Explanation:** What we do here is compare both models using several key metrics, showing both a table with the values and a bar chart to visually see which one performs better in each area. This makes it easier to identify the strengths and weaknesses of each model more clearly than just looking at individual numbers.

- **Improvement:** I removed the print statements that didn't add much value and organized the code to make it cleaner. I also reused what had already been calculated from the metrics so as not to repeat logic unnecessarily.

In [ ]:
def create_feature_dimensionality_comparison_table(title, dim_data):
    table = Table(title=title)

    table.add_column("Approach", justify="center", style="cyan")
    table.add_column("Feature Dimensionality", justify="right", style="green")
    table.add_column("Dimensionality Reduction (%)", justify="right", style="green")

    for i, approach in enumerate(dim_data["Approach"]):
        table.add_row(
            approach,
            f'{dim_data["Feature Dimensionality"][i]}',
            f'{dim_data["Dimensionality Reduction (%)"][i]:.2f}',
        )

    console.print(table)


def plot_feature_dimensionality_comparison(title, dim_data):

    plt.figure(figsize=(10, 6))
    plt.bar(
        dim_data["Approach"],
        dim_data["Feature Dimensionality"],
        color=["skyblue", "orange"],
    )
    plt.title(title)
    plt.ylabel("Number of Features")
    plt.yscale("log")
    plt.grid(axis="y", linestyle="--", alpha=0.7)

    for i, v in enumerate(dim_data["Feature Dimensionality"]):
        plt.text(i, v * 1.1, f"{int(v)}", ha="center")

    plt.tight_layout()
    plt.show()


dim_data = {
    "Approach": [CLASSIFIER_A, CLASSIFIER_B],
    "Feature Dimensionality": [X_train_direct.shape[1], X_train_alexnet.shape[1]],
    "Dimensionality Reduction (%)": [
        0,
        (1 - X_train_alexnet.shape[1] / X_train_direct.shape[1]) * 100,
    ],
}

create_feature_dimensionality_comparison_table(
    "Feature Dimensionality Comparison", dim_data
)
plot_feature_dimensionality_comparison("Feature Dimensionality Comparison", dim_data)

- **Explanation:** What we do here is compare the number of features between the direct approach and AlexNet. The number of features and the percentage reduction are shown in a table, and then visualized in a bar chart with a logarithmic scale to highlight the difference.

- **Improvement:** I replaced the print statements with a table and separated the logic into two functions (table and graph), making the code clearer and more reusable.

In [ ]:
def plot_learning_curves(results_direct, results_alexnet):
    plt.figure(figsize=(12, 6))
    plt.plot(results_direct["loss_curve"], label=CLASSIFIER_A, color="blue")
    plt.plot(results_alexnet["loss_curve"], label=CLASSIFIER_B, color="orange")
    plt.xlabel("Iterations")
    plt.ylabel("Loss")
    plt.title("Learning Curves Comparison")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()


def create_convergence_iterations_table(convergence_analysis):
    table = Table(
        title=f"Iterations to Converge - Convergence Speed: {convergence_analysis}"
    )

    table.add_column("Model", justify="center", style="cyan")
    table.add_column("Iterations", justify="center", style="green")

    table.add_row("Direct Approach", str(len(results_direct["loss_curve"])))
    table.add_row("AlexNet", str(len(results_alexnet["loss_curve"])))

    console.print(table)


len_alexnet_loss_curve = len(results_alexnet["loss_curve"])
len_direct_loss_curve = len(results_direct["loss_curve"])

convergence_analysis = (
    "Faster" if len_alexnet_loss_curve < len_direct_loss_curve else "Slower"
)

plot_learning_curves(results_direct, results_alexnet)
create_convergence_iterations_table(convergence_analysis)

- **Explanation:** What we do here is plot the learning curve of both models to see how the loss decreases in each iteration. We also compare how many iterations each one needed to converge, showing that AlexNet does it faster.

- **Improvement:** I created a method to generate a comparison table and separated the convergence variables to make the code clearer.

In [ ]:
def create_computational_comparison_table():
    table = Table(title="Computational Requirements Comparison")

    table.add_column("Metric", justify="left", style="cyan")
    table.add_column(f"{CLASSIFIER_A}", justify="center", style="green")
    table.add_column(f"{CLASSIFIER_B}", justify="center", style="green")

    metrics = [
        (
            "Feature Extraction Time",
            "Lower - Simple resizing and flattening",
            "Higher - Forward pass through AlexNet",
        ),
        (
            "Model Training Time",
            f"{results_direct['training_time']:.2f} seconds",
            f"{results_alexnet['training_time']:.2f} seconds",
        ),
        (
            "Memory Usage (Features)",
            f"{X_train_direct.nbytes / (1024 * 1024):.2f} MB",
            f"{X_train_alexnet.nbytes / (1024 * 1024):.2f} MB",
        ),
        (
            "Preprocessing Requirements",
            "Minimal - Resize and normalize",
            "More complex - Requires RGB conversion",
        ),
        (
            "Total Parameters",
            f"{X_train_direct.shape[1] + 1}",
            f"{X_train_alexnet.shape[1] + 1}",
        ),
    ]

    for metric, classifier_a_value, classifier_b_value in metrics:
        table.add_row(metric, classifier_a_value, classifier_b_value)

    console.print(table)


create_computational_comparison_table()

- **Explanation:** What we do here is compare the computational requirements of both approaches using a printout, including feature extraction time, training time, memory usage, preprocessing requirements, and number of parameters.

- **Improvement:** Create a method to generate a comparative table to make it easier to visualize the data.

In [ ]:
def calculate_improvements(
    results_direct, results_alexnet, X_train_direct, X_train_alexnet
):

    direct_acc = results_direct["test_accuracy"]
    alexnet_acc = results_alexnet["test_accuracy"]

    acc_diff = alexnet_acc - direct_acc
    accuracy_improvement = (acc_diff / direct_acc) * 100

    direct_time = results_direct["training_time"]
    alexnet_time = results_alexnet["training_time"]

    time_diff = direct_time - alexnet_time
    training_time_improvement = (time_diff / direct_time) * 100

    direct_dim = X_train_direct.shape[1]
    alexnet_dim = X_train_alexnet.shape[1]

    dim_diff = direct_dim - alexnet_dim
    dimensionality_reduction = (dim_diff / direct_dim) * 100

    return {
        "accuracy_improvement": accuracy_improvement,
        "training_time_improvement": training_time_improvement,
        "dimensionality_reduction": dimensionality_reduction,
    }


def create_improvements_table(improvements):
    table = Table(title="Relative Improvements (AlexNet vs Direct)")

    table.add_column("Metric", justify="left", style="cyan")
    table.add_column("Improvement (%)", justify="center", style="green")

    metrics = [
        ("Accuracy Improvement", f"{improvements['accuracy_improvement']:.2f}%"),
        (
            "Training Time Improvement",
            f"{improvements['training_time_improvement']:.2f}%",
        ),
        (
            "Dimensionality Reduction",
            f"{improvements['dimensionality_reduction']:.2f}%",
        ),
    ]

    for metric, value in metrics:
        table.add_row(metric, value)

    console.print(table)


def plot_improvements(
    accuracy_improvement, training_time_improvement, dimensionality_reduction
):
    plt.figure(figsize=(12, 6))

    improvements = [
        "Accuracy Improvement (%)",
        "Training Time Improvement (%)",
        "Dimensionality Reduction (%)",
    ]
    values = [accuracy_improvement, training_time_improvement, dimensionality_reduction]
    colors = ["green" if v > 0 else "red" for v in values]

    plt.bar(improvements, values, color=colors)
    plt.axhline(0, color="black", linestyle="-", alpha=0.3)
    plt.title("Transfer Learning Improvements")
    plt.ylabel("Percentage (%)")
    plt.grid(axis="y", linestyle="--", alpha=0.3)

    for i, v in enumerate(values):
        plt.text(i, v + np.sign(v) * 5, f"{v:.2f}%", ha="center", color="black")

    plt.tight_layout()
    plt.show()


improvements = calculate_improvements(
    results_direct, results_alexnet, X_train_direct, X_train_alexnet
)

create_improvements_table(improvements)
plot_improvements(
    improvements["accuracy_improvement"],
    improvements["training_time_improvement"],
    improvements["dimensionality_reduction"],
)

- **Explanation:** Here, three improvements that AlexNet had over the direct approach are calculated: increased accuracy, reduced training time, and reduced dimensionality. These values are first printed to the console and then displayed in a bar chart.

- **Improvement:** I separated the logic into different methods, one to calculate the improvements, another to display them in a table, and another to graph them. This makes the code more organized and easier to read.

# Conclusions

This analysis demonstrated that transfer learning provides significant improvements in both performance and computational efficiency. AlexNet not only achieved greater accuracy, but also reduced data dimensionality and accelerated convergence compared to the direct approach. This confirms the importance of reusing pre-trained models, especially when working with limited datasets or complex problems.

Likewise, best practices for transfer learning should be taken into account:

- Use differential learning rates: lower layers with small values and upper layers with higher values.
- Avoid retraining layers if the dataset is small and similar to the original; simply freeze and add a classifier.
- When working with larger datasets, consider full fine-tuning by replacing the last layer.
- Be sure to choose known and proven architectures for the domain in question.